# VAE(Variational Auto Encoder) 実践的学習教材

このノートブックでは、VAE(Variational Auto Encoder / 変分オートエンコーダ)を、数式・PyTorch実装・実験の順に学びます。単なる画像圧縮ではなく、「潜在変数からデータを生成するモデル」としてVAEを扱います。

扱う内容:

- Auto EncoderとVAEの違い
- 潜在変数モデルとELBOの直感
- reparameterization trickの実装
- MNIST/FashionMNISTを使ったVAEの学習
- 再構成画像、ランダム生成、潜在空間の可視化
- β-VAEによる再構成品質と潜在空間のトレードオフ
- 実務で使うときの設計・評価の観点

CPUでも動くように小さめのMLPモデルから始めます。GPUが使える場合は自動で利用します。

## 0. 依存関係

この教材では `torch`, `torchvision`, `matplotlib`, `numpy`, `pandas` を使います。

```bash
pip install torch torchvision matplotlib numpy pandas
```

初回実行時はMNISTまたはFashionMNISTを `./data` にダウンロードします。ネットワークが使えない環境では、すでにデータがキャッシュされている必要があります。

In [ ]:
import math
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 1. Auto EncoderからVAEへ

通常のAuto Encoderは、入力 $x$ をエンコーダで低次元表現 $z$ に圧縮し、デコーダで元の $x$ を復元します。

$$
z = f_\phi(x), \quad \hat{x} = g_\theta(z)
$$

ここでの $z$ は、入力画像ごとに決まる1つの点です。たとえば2次元潜在空間なら、ある数字の画像は `(1.2, -0.7)` のような1点に写されます。

Auto Encoderは再構成には便利ですが、潜在空間の形は自由です。訓練データが写された点の周辺だけをデコーダが学習するため、潜在空間の空いている場所から適当に $z$ を選んでも、自然な画像が出るとは限りません。極端に言えば、学習済みの点がバラバラに散らばっていて、その間の領域がデコーダにとって未知の場所になっている可能性があります。

VAEでは、エンコーダが1点の $z$ ではなく、潜在変数の分布を出力します。具体的には、入力 $x$ に対して平均 $\mu$ と分散 $\sigma^2$ を出し、その正規分布から $z$ をサンプリングします。

$$
q_\phi(z|x) = \mathcal{N}(\mu_\phi(x), \operatorname{diag}(\sigma_\phi(x)^2))
$$

ここで「潜在分布」という言葉は混乱しやすいので、2種類に分けて考えます。

- **各入力ごとの潜在分布**: 1枚の画像 $x$ を入れたときに出る $q_\phi(z|x)$ です。これは「この画像に対応する $z$ は、この平均と分散の周辺にありそう」という分布です。
- **学習データ全体で見た潜在表現の分布**: すべての学習画像をエンコーダに通したとき、$\mu$ やサンプルされた $z$ が潜在空間全体にどう散らばるかです。

VAEのKL損失で直接計算しているのは、前者の **各入力ごとの潜在分布 $q_\phi(z|x)$** と標準正規分布 $p(z)$ の距離です。ただし、それを全学習データに対して毎回かけるため、結果として後者の「学習データ全体で見た $z$ の散らばり」も標準正規分布の領域に収まりやすくなります。

つまり、VAEのエンコーダは「この画像は潜在空間のこの1点です」とは言いません。「この画像は、この平均と分散を持つ正規分布の周辺にあります」と表現します。同じ画像を入れても、学習中はその分布から少し揺らいだ $z$ が使われます。

さらにVAEでは、各入力から作られる分布 $q_\phi(z|x)$ を、標準正規分布 $p(z)$ から大きく外れすぎないようにします。

$$
p(z) = \mathcal{N}(0, I)
$$

ここで大事なのは、「入力画像そのものが正規分布に従う」という意味ではないことです。正規分布に近づける対象は、画像データではなく、エンコーダが内部で作る潜在変数 $z$ の分布です。

なぜ標準正規分布に近づけるのでしょうか。生成時には入力画像がないため、エンコーダを使えません。その代わりに、私たちは $z \sim \mathcal{N}(0, I)$ としてランダムに潜在変数を作り、デコーダに渡します。もし各入力ごとの $q_\phi(z|x)$ が標準正規分布から大きく外れないように学習されていれば、学習データ全体で見た $z$ の散らばりも標準正規分布の周辺に収まりやすくなります。その結果、生成時にランダムに選んだ $z$ も、デコーダが見慣れた領域に入りやすくなります。

この制約は、潜在空間をなめらかにする効果もあります。近い $z$ からは似た画像が生成されやすくなり、2つの画像の潜在表現を補間したときにも、途中で急に壊れた画像になりにくくなります。

まとめると、Auto Encoderは「よく復元できる圧縮表現」を学び、VAEはそれに加えて「標準正規分布からサンプリングして生成しやすい潜在空間」を学びます。この違いが、VAEを生成モデルとして使える理由です。

## 2. VAEの目的関数: 再構成 + KL正則化

VAEでは次の損失を最小化します。

$$
\mathcal{L}(x) = \underbrace{-\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]}_{\text{reconstruction loss}} + \beta \underbrace{D_{KL}(q_\phi(z|x) \| p(z))}_{\text{KL loss}}
$$

この式は難しく見えますが、実装では大きく2つの部品に分かれます。

1つ目は再構成損失です。これは「入力画像 $x$ と、デコーダが作った画像 $\hat{x}$ がどれくらい違うか」を測ります。この教材ではMNISTのような白黒画像を扱うため、各画素を0から1の値として表します。黒に近い画素は0、白に近い画素は1、中間の灰色は0.3や0.7のような値です。`transforms.ToTensor()` を使うと、元の0〜255の画素値が0〜1に変換されます。

デコーダの最後に `sigmoid` を置いているのもこのためです。`sigmoid` の出力は必ず0〜1になるので、各画素について「白である確率」のように解釈できます。この解釈を使うと、再構成損失としてBinary Cross Entropy(BCE)を使えます。BCEは、正解の画素値が1に近い場所では出力も1に近く、正解が0に近い場所では出力も0に近くなるように罰を与えます。

厳密にはMNISTの画素は完全な0/1だけではなく灰色も含みます。それでも、0〜1の値を「白さの度合い」として扱うと、BCEは実用上よく使える再構成損失になります。連続値画像をより自然に扱いたい場合は、デコーダ出力をガウス分布の平均とみなし、MSEを使う設計もあります。

2つ目はKL損失です。これは「1つの入力 $x$ に対してエンコーダが作った分布 $q_\phi(z|x)$ が、標準正規分布 $p(z)$ からどれくらい離れているか」を測ります。つまりKL損失が直接見ているのは、学習データ全体の $z$ の分布ではなく、各サンプルごとの $q_\phi(z|x)$ です。ただし、この損失をすべての学習サンプルに対して平均するため、結果としてデータ全体で見た潜在表現の散らばりも標準正規分布の周辺に寄りやすくなります。

ここで出てくる「閉形式で計算できる」とは、サンプリングや数値積分で近似しなくても、$\mu$ と $\sigma^2$ を式に代入するだけでKL損失を直接計算できる、という意味です。たとえば「この分布と標準正規分布のずれを大量の乱数で推定する」のではなく、次の式をそのままPyTorchで計算できます。

$$
D_{KL}(\mathcal{N}(\mu, \sigma^2) \| \mathcal{N}(0, 1))
= -\frac{1}{2}\sum_j (1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2)
$$

この式の $j$ は潜在次元を表します。`latent_dim=2` なら2次元ぶんを足し、`latent_dim=32` なら32次元ぶんを足します。実装では分散 $\sigma^2$ を直接出す代わりに `logvar = log(sigma^2)` を出すことが多いため、コードでは次の形になります。

```python
kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
```

このKL損失は、$\mu$ が0から遠いほど大きくなり、$\sigma^2$ が1から大きく外れても大きくなります。つまり、エンコーダに対して「平均は0の近く、分散は1の近くにしすぎてほしい。ただし再構成に必要な情報は残してよい」という圧力をかけます。

$\beta=1$ が標準的なVAEです。$\beta$ を大きくすると潜在空間はきれいになりやすい一方で、再構成画像はぼやけやすくなります。

## 3. Reparameterization Trick

エンコーダは $\mu$ と $\log\sigma^2$ を出力します。そこから直接サンプリングすると、乱数サンプリングの部分で勾配が流れにくくなります。

そこで、乱数 $\epsilon \sim \mathcal{N}(0, I)$ を外から取り、次の形に変形します。

$$
z = \mu + \sigma \odot \epsilon
$$

これにより、確率的なノイズは $\epsilon$ に分離され、$\mu$ と $\sigma$ には通常の微分可能な演算として勾配が流れます。

In [ ]:
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + std * eps


mu = torch.zeros(4, 2)
logvar = torch.zeros(4, 2)
z = reparameterize(mu, logvar)
z

## 4. データセットを準備する

まずはMNISTを使います。`dataset_name = "FashionMNIST"` に変えると、少し難しい衣類画像でも同じ実験ができます。

学習時間を短くするため、デフォルトでは学習データの一部だけを使います。精度や生成品質を上げたい場合は `train_size` と `epochs` を増やしてください。

In [ ]:
@dataclass
class VAEConfig:
    dataset_name: str = "MNIST"  # "MNIST" or "FashionMNIST"
    data_dir: str = "./data"
    train_size: int = 12000
    test_size: int = 2000
    batch_size: int = 128
    latent_dim: int = 2
    hidden_dim: int = 400
    lr: float = 1e-3
    epochs: int = 8
    beta: float = 1.0


cfg = VAEConfig()

transform = transforms.Compose([
    transforms.ToTensor(),
])

dataset_cls = datasets.MNIST if cfg.dataset_name == "MNIST" else datasets.FashionMNIST
train_dataset_full = dataset_cls(cfg.data_dir, train=True, download=True, transform=transform)
test_dataset_full = dataset_cls(cfg.data_dir, train=False, download=True, transform=transform)

train_indices = torch.randperm(len(train_dataset_full))[:cfg.train_size]
test_indices = torch.arange(min(cfg.test_size, len(test_dataset_full)))

train_dataset = Subset(train_dataset_full, train_indices)
test_dataset = Subset(test_dataset_full, test_indices)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

len(train_dataset), len(test_dataset)

In [ ]:
def show_image_grid(images, nrow=8, title=None):
    images = images.detach().cpu().clamp(0, 1)
    n = min(len(images), nrow * nrow)
    rows = math.ceil(n / nrow)
    fig, axes = plt.subplots(rows, nrow, figsize=(1.4 * nrow, 1.4 * rows))
    axes = np.array(axes).reshape(rows, nrow)
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < n:
            ax.imshow(images[i].squeeze(), cmap="gray", vmin=0, vmax=1)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


sample_images, sample_labels = next(iter(train_loader))
show_image_grid(sample_images[:32], nrow=8, title=f"{cfg.dataset_name} samples")

## 5. MLP VAEを実装する

最初のモデルは28x28画像を784次元ベクトルに平坦化して処理します。

- encoder: 画像 $x$ から $\mu$ と $\log\sigma^2$ を出力
- sampling: reparameterization trickで $z$ を作る
- decoder: $z$ から画像 $\hat{x}$ を生成

デコーダ出力には `sigmoid` を使い、0から1の画素値として解釈します。

In [ ]:
class MLPVAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=2):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x.view(x.size(0), -1))
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z):
        recon = self.decoder(z)
        return recon.view(-1, 1, 28, 28)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z


model = MLPVAE(hidden_dim=cfg.hidden_dim, latent_dim=cfg.latent_dim).to(device)
sum(p.numel() for p in model.parameters())

## 6. 損失関数を実装する

再構成損失は画像全体のBCEをバッチ平均します。KL損失もサンプルごとに合計してからバッチ平均します。

VAEでは、総損失だけでなく `recon_loss` と `kl_loss` を分けて記録することが重要です。再構成だけが下がり、KLがほぼ0になる場合は潜在変数を使っていない可能性があります。逆にKLが強すぎると画像がぼやけます。

In [ ]:
def vae_loss(recon, x, mu, logvar, beta=1.0):
    batch_size = x.size(0)
    recon_loss = F.binary_cross_entropy(recon, x, reduction="sum") / batch_size
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size
    loss = recon_loss + beta * kl_loss
    return loss, recon_loss, kl_loss


xb, _ = next(iter(train_loader))
xb = xb.to(device)
recon, mu, logvar, z = model(xb)
loss, recon_loss, kl_loss = vae_loss(recon, xb, mu, logvar, beta=cfg.beta)
loss.item(), recon_loss.item(), kl_loss.item()

## 7. 学習ループ

VAEの学習では、エポックごとに次の値を観察します。

- `loss`: 全体の目的関数
- `recon`: 入力をどれだけ復元できているか
- `kl`: 潜在分布が標準正規分布からどれだけ離れているか

2次元潜在空間は表現力が低い代わりに可視化しやすい設定です。生成品質を優先する場合は `latent_dim=16` や `latent_dim=32` に増やしてください。

In [ ]:
def run_epoch(model, loader, optimizer=None, beta=1.0):
    is_train = optimizer is not None
    model.train(is_train)
    totals = {"loss": 0.0, "recon": 0.0, "kl": 0.0, "n": 0}

    for x, _ in loader:
        x = x.to(device)
        if is_train:
            optimizer.zero_grad(set_to_none=True)

        recon, mu, logvar, _ = model(x)
        loss, recon_loss, kl_loss = vae_loss(recon, x, mu, logvar, beta=beta)

        if is_train:
            loss.backward()
            optimizer.step()

        batch_size = x.size(0)
        totals["loss"] += loss.item() * batch_size
        totals["recon"] += recon_loss.item() * batch_size
        totals["kl"] += kl_loss.item() * batch_size
        totals["n"] += batch_size

    return {k: totals[k] / totals["n"] for k in ["loss", "recon", "kl"]}


def train_vae(model, train_loader, test_loader, epochs=8, lr=1e-3, beta=1.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, train_loader, optimizer=optimizer, beta=beta)
        with torch.no_grad():
            test_metrics = run_epoch(model, test_loader, optimizer=None, beta=beta)

        row = {"epoch": epoch, "split": "train", **train_metrics}
        history.append(row)
        row = {"epoch": epoch, "split": "test", **test_metrics}
        history.append(row)

        print(
            f"epoch {epoch:02d} | "
            f"train loss {train_metrics['loss']:.2f} recon {train_metrics['recon']:.2f} kl {train_metrics['kl']:.2f} | "
            f"test loss {test_metrics['loss']:.2f} recon {test_metrics['recon']:.2f} kl {test_metrics['kl']:.2f}"
        )
    return pd.DataFrame(history)


history = train_vae(model, train_loader, test_loader, epochs=cfg.epochs, lr=cfg.lr, beta=cfg.beta)

In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    for ax, metric in zip(axes, ["loss", "recon", "kl"]):
        for split, df in history.groupby("split"):
            ax.plot(df["epoch"], df[metric], marker="o", label=split)
        ax.set_title(metric)
        ax.set_xlabel("epoch")
        ax.grid(True)
        ax.legend()
    plt.tight_layout()
    plt.show()


plot_history(history)

## 8. 再構成画像を見る

上段が入力画像、下段がVAEの再構成画像です。VAEは潜在空間を正規分布に近づける制約を持つため、通常のAuto Encoderより再構成が少しぼやけることがあります。

In [ ]:
@torch.no_grad()
def show_reconstructions(model, loader, n=12):
    model.eval()
    x, _ = next(iter(loader))
    x = x[:n].to(device)
    recon, _, _, _ = model(x)
    stacked = torch.cat([x.cpu(), recon.cpu()], dim=0)
    show_image_grid(stacked, nrow=n, title="top: input, bottom: reconstruction")


show_reconstructions(model, test_loader, n=12)

## 9. 潜在空間を可視化する

`latent_dim=2` にしているため、各画像の平均ベクトル $\mu$ を2次元散布図として表示できます。点の色はラベルです。

きれいに学習できている場合、同じ数字や似た形の数字が近い場所に集まりやすくなります。ただし、VAEは分類器ではないため、クラスを完全に分離することが目的ではありません。

In [ ]:
@torch.no_grad()
def collect_latents(model, loader, max_batches=20):
    model.eval()
    all_mu = []
    all_y = []
    for i, (x, y) in enumerate(loader):
        if i >= max_batches:
            break
        x = x.to(device)
        mu, logvar = model.encode(x)
        all_mu.append(mu.cpu())
        all_y.append(y)
    return torch.cat(all_mu), torch.cat(all_y)


latents, labels = collect_latents(model, test_loader)

plt.figure(figsize=(7, 6))
scatter = plt.scatter(latents[:, 0], latents[:, 1], c=labels, s=8, cmap="tab10", alpha=0.75)
plt.colorbar(scatter, ticks=range(10))
plt.xlabel("z[0] mean")
plt.ylabel("z[1] mean")
plt.title("latent space by encoder mean")
plt.grid(True)
plt.show()

## 10. 潜在空間をグリッドで歩く

2次元潜在空間上の格子点をデコーダに入力し、生成される画像を並べます。近い座標で似た画像が出ていれば、潜在空間が連続的に学習されています。

In [ ]:
@torch.no_grad()
def show_latent_grid(model, grid_size=15, z_range=3.0):
    if model.latent_dim != 2:
        print("latent_dim=2 のモデルで実行してください。")
        return
    model.eval()
    xs = torch.linspace(-z_range, z_range, grid_size)
    ys = torch.linspace(z_range, -z_range, grid_size)
    z = torch.tensor([[x, y] for y in ys for x in xs], device=device)
    images = model.decode(z).cpu()

    canvas = torch.zeros(grid_size * 28, grid_size * 28)
    for idx, img in enumerate(images):
        row = idx // grid_size
        col = idx % grid_size
        canvas[row * 28:(row + 1) * 28, col * 28:(col + 1) * 28] = img.squeeze()

    plt.figure(figsize=(8, 8))
    plt.imshow(canvas, cmap="gray", vmin=0, vmax=1)
    plt.axis("off")
    plt.title("decoder output over 2D latent grid")
    plt.show()


show_latent_grid(model, grid_size=15, z_range=3.0)

## 11. ランダムサンプリングで画像を生成する

VAEの生成時は、エンコーダを使わずに $z \sim \mathcal{N}(0, I)$ をサンプリングし、デコーダに入力します。

通常のAuto Encoderではこの操作がうまくいく保証はありません。VAEが潜在分布を標準正規分布に近づけていることが、生成に効いています。

In [ ]:
@torch.no_grad()
def sample_from_prior(model, n=32):
    model.eval()
    z = torch.randn(n, model.latent_dim, device=device)
    samples = model.decode(z)
    show_image_grid(samples, nrow=8, title="samples from z ~ N(0, I)")


sample_from_prior(model, n=32)

## 12. 補間: 2つの画像の間を移動する

2つの入力画像をエンコードして得た $\mu_1, \mu_2$ の間を線形補間し、デコーダに通します。潜在空間が連続的なら、画像が徐々に変化します。

In [ ]:
@torch.no_grad()
def interpolate_images(model, loader, steps=12, idx_a=0, idx_b=1):
    model.eval()
    x, y = next(iter(loader))
    a = x[idx_a:idx_a + 1].to(device)
    b = x[idx_b:idx_b + 1].to(device)
    mu_a, _ = model.encode(a)
    mu_b, _ = model.encode(b)

    weights = torch.linspace(0, 1, steps, device=device).unsqueeze(1)
    z = (1 - weights) * mu_a + weights * mu_b
    images = model.decode(z)
    show_image_grid(images, nrow=steps, title=f"latent interpolation: label {int(y[idx_a])} to {int(y[idx_b])}")


interpolate_images(model, test_loader, steps=12, idx_a=0, idx_b=9)

## 13. β-VAEでトレードオフを観察する

$\beta$ を変えると、再構成と潜在空間の正則化のバランスが変わります。

- 小さい $\beta$: 再構成を優先しやすいが、潜在空間が標準正規分布から離れやすい
- 大きい $\beta$: 潜在空間は整いやすいが、再構成が粗くなりやすい

下の実験は追加学習なので少し時間がかかります。まず全体を軽く試したい場合は `run_beta_experiment = False` のままにしてください。

In [ ]:
run_beta_experiment = False
beta_values = [0.2, 1.0, 4.0]
beta_epochs = 5

beta_results = []
beta_models = {}

if run_beta_experiment:
    for beta in beta_values:
        print(f"\nTraining beta={beta}")
        set_seed(42)
        m = MLPVAE(hidden_dim=cfg.hidden_dim, latent_dim=cfg.latent_dim).to(device)
        h = train_vae(m, train_loader, test_loader, epochs=beta_epochs, lr=cfg.lr, beta=beta)
        final = h[(h["split"] == "test") & (h["epoch"] == beta_epochs)].iloc[0].to_dict()
        final["beta"] = beta
        beta_results.append(final)
        beta_models[beta] = m

    beta_df = pd.DataFrame(beta_results)
    display(beta_df[["beta", "loss", "recon", "kl"]])
else:
    print("Set run_beta_experiment = True to compare beta values.")

In [ ]:
if run_beta_experiment:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(beta_df["beta"], beta_df["recon"], marker="o")
    axes[0].set_xscale("log")
    axes[0].set_title("reconstruction loss")
    axes[0].set_xlabel("beta")
    axes[0].grid(True)

    axes[1].plot(beta_df["beta"], beta_df["kl"], marker="o")
    axes[1].set_xscale("log")
    axes[1].set_title("KL loss")
    axes[1].set_xlabel("beta")
    axes[1].grid(True)
    plt.tight_layout()
    plt.show()

    for beta, m in beta_models.items():
        print(f"beta={beta}")
        show_reconstructions(m, test_loader, n=10)
        sample_from_prior(m, n=16)

## 14. Conv VAEへの発展

画像ではMLPより畳み込みを使う方が自然です。MLP VAEは理解しやすい一方で、画像の局所構造を明示的には使いません。実務や高品質生成では、畳み込みエンコーダ・デコーダ、またはU-Net系の構造を使うことが多いです。

以下はConv VAEの最小例です。学習ループと損失関数はMLP VAEと同じものを使えます。`latent_dim` を2より大きくすると、再構成は改善しやすくなります。

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),  # 14x14
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # 7x7
            nn.ReLU(),
            nn.Flatten(),
        )
        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, 64 * 7 * 7)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # 14x14
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),  # 28x28
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z):
        h = self.fc_dec(z).view(-1, 64, 7, 7)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z


run_conv_vae = False
if run_conv_vae:
    conv_model = ConvVAE(latent_dim=16).to(device)
    conv_history = train_vae(conv_model, train_loader, test_loader, epochs=8, lr=1e-3, beta=1.0)
    plot_history(conv_history)
    show_reconstructions(conv_model, test_loader, n=12)
    sample_from_prior(conv_model, n=32)
else:
    print("Set run_conv_vae = True to train a small convolutional VAE.")

## 15. 実務でのチェックポイント

VAEを実務で使うときは、生成画像だけでなく損失の内訳と潜在空間の使われ方を確認します。

| 観点 | 見るもの | よくある問題 |
|---|---|---|
| 再構成品質 | 入力と再構成の比較、BCE/MSE | ぼやける、細部が消える |
| 潜在空間 | $\mu$ の分布、補間、サンプリング | 穴が多い、近い点で画像が急変する |
| KL項 | `kl_loss` の推移 | ほぼ0ならposterior collapseの可能性 |
| 生成品質 | priorからのサンプル | 訓練画像に似ない、同じような画像ばかり出る |
| 目的との一致 | 異常検知、圧縮、生成、表現学習 | 損失が下がってもタスク性能が上がらない |

画像生成の品質だけを追求するなら、現代ではDiffusionモデルやGANが使われることも多いです。一方でVAEは、潜在変数の扱いや確率的生成モデルの基礎を学ぶ教材として非常に有用で、異常検知・表現学習・生成モデルの前処理にも応用できます。

## 16. 演習

1. `latent_dim` を `2, 8, 32` に変えて、再構成画像とpriorサンプルを比較してください。
2. `dataset_name` を `FashionMNIST` に変えて、MNISTより難しくなる点を観察してください。
3. `beta` を `0.1, 1.0, 5.0` に変えて、`recon_loss` と `kl_loss` の関係を説明してください。
4. `run_conv_vae = True` にして、MLP VAEとConv VAEの再構成画像を比較してください。
5. 再構成誤差が大きいサンプルを抽出し、どんな画像で失敗しやすいか確認してください。

発展課題: ラベルを条件としてデコーダに渡すConditional VAEを実装し、「指定した数字」を生成できるようにしてください。

## 17. まとめ

このノートブックでは、VAEを次の流れで実装しました。

- エンコーダが $\mu$ と $\log\sigma^2$ を出す
- reparameterization trickで微分可能なサンプリングを行う
- デコーダが潜在変数から画像を復元・生成する
- 損失は再構成損失とKL正則化の和で構成する
- priorからサンプリングすることで新しい画像を生成できる

VAEの本質は、Auto Encoderに確率的な潜在空間を導入し、生成可能な表現を学習することです。再構成品質だけでなく、潜在空間が連続的で、標準正規分布から自然にサンプリングできるかを観察することが重要です。